<a href="https://colab.research.google.com/github/Prakruthi2606/AI_SQL_Data_Agent/blob/main/AI_SQL_Data_analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dependencies**

In [ ]:
!pip install pandas matplotlib plotly sqlalchemy langchain langchain-community langchain-groq streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


# **Libraries**

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import plotly.express as px

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_groq import ChatGroq

# **Setting up API**

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key"

# **Load and preview**

In [ ]:
file_path = "train.csv"   # change this

df = pd.read_csv(file_path)

print("Preview Data:")
display(df.head())

print("\nData Info:")
print(df.info())

Preview Data:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
d

# **Data Profiling**

In [ ]:
print("Column Names:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nSummary:\n", df.describe())

Column Names: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']

Missing Values:
 Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

Summary:
             Row ID   Postal Code         Sales
count  9800.000000   9789.000000   9800.000000
mean   4900.500000  55273.322403    230.769059
std    2829.160653  32041.223413    626.651875
min       1.000000   1040.000000      0.444000
25%    2450.750000  23223.000000     17.248000
50%    4900.500000  58103.000000     54.490000
75%    7350.250000  90008.000

# **Convert CSV → SQLite**

In [ ]:
conn = sqlite3.connect("data.db")

table_name = "data_table"

df.to_sql(table_name, conn, if_exists="replace", index=False)

print("Database created successfully!")

Database created successfully!


# **Connecting Langchain to DB**

In [ ]:
db = SQLDatabase.from_uri("sqlite:///data.db")

print("Tables in DB:", db.get_usable_table_names())

Tables in DB: ['data_table']


# **Initialize LLM**

In [ ]:
from langchain_groq import ChatGroq

def get_llm():
    models = [
        "llama-3.3-70b-versatile",  # primary
        "mixtral-8x7b-32768",       # backup
        "llama-3.1-8b-instant"      # lightweight
    ]

    for m in models:
        try:
            llm = ChatGroq(model=m, temperature=0)
            # quick test call (small)
            llm.invoke("ping")
            print(f"✅ Using model: {m}")
            return llm
        except Exception:
            continue

    raise Exception("❌ No working Groq model found")

llm = get_llm()

✅ Using model: llama-3.3-70b-versatile


# **Create SQL Agent**

In [ ]:
agent = create_sql_agent(
    llm=llm,
    db=db,
    verbose=False
)

# **Asking questions**

In [ ]:
query = input("Ask your question: ")

TABLE_NAME = "data_table"

# ---------------------------
# STEP 1: Generate SQL
# ---------------------------
columns = df.columns.tolist()
schema = "Columns:\n" + "\n".join([f"- {col}" for col in columns])

sql_prompt = f"""
You are an expert SQL generator.

Table: {TABLE_NAME}

{schema}

Rules:
- ALWAYS use table name: {TABLE_NAME}
- Use exact column names
- Wrap columns with spaces in double quotes
- Return ONLY SQL
- No markdown

Question: {query}
"""

raw_sql = llm.invoke(sql_prompt).content
sql_query = raw_sql.replace("```sql", "").replace("```", "").strip()

print("\n🧾 SQL Query:")
print(sql_query)

# ---------------------------
# STEP 2: Execute SQL
# ---------------------------
try:
    result_df = pd.read_sql_query(sql_query, conn)

    print("\n📊 Result:")
    print(result_df)

except Exception as e:
    print("\n❌ SQL Error:", e)
    result_df = None


# ---------------------------
# STEP 3: Visualization
# ---------------------------
import plotly.express as px

def auto_plot(data):
    if data is None or data.empty:
        print("\n⚠️ No data to plot")
        return

    # Skip plot if only one value
    if data.shape[1] == 1:
        print("\n📊 Single value (no plot needed)")
        return

    numeric_cols = data.select_dtypes(include=['number']).columns
    categorical_cols = data.select_dtypes(include=['object']).columns

    if len(numeric_cols) >= 1 and len(categorical_cols) >= 1:
        fig = px.bar(
            data,
            x=categorical_cols[0],
            y=numeric_cols[0],
            title="📊 Visualization",
            template="plotly_dark"
        )
        fig.show()

    elif len(numeric_cols) >= 2:
        fig = px.scatter(
            data,
            x=numeric_cols[0],
            y=numeric_cols[1],
            title="📊 Visualization",
            template="plotly_dark"
        )
        fig.show()

auto_plot(result_df)


# ---------------------------
# STEP 4: AI FINAL ANSWER
# ---------------------------
if result_df is not None:
    answer_prompt = f"""
    Answer clearly based on the result.

    Question: {query}
    Result: {result_df.head(5).to_string(index=False)}
    """

    final_answer = llm.invoke(answer_prompt)

    print("\n🤖 AI Answer:")
    print(final_answer.content)

Ask your question: average sales per state

🧾 SQL Query:
SELECT "State", AVG(Sales) AS average_sales FROM data_table GROUP BY "State"

📊 Result:
                   State  average_sales
0                Alabama     319.846557
1                Arizona     158.173350
2               Arkansas     194.635500
3             California     229.345562
4               Colorado     177.886022
5            Connecticut     163.223866
6               Delaware     293.795688
7   District of Columbia     286.502000
8                Florida     237.095260
9                Georgia     272.424350
10                 Idaho     208.689810
11              Illinois     164.050760
12               Indiana     360.877037
13                  Iowa     170.906154
14                Kansas     121.429583
15              Kentucky     266.119635
16             Louisiana     222.708537
17                 Maine     158.816250
18              Maryland     225.766886
19         Massachusetts     212.106919
20             


🤖 AI Answer:
The average sales per state are as follows:

1. Alabama: $319.85
2. Arizona: $158.17
3. Arkansas: $194.64
4. California: $229.35
5. Colorado: $177.89
